QR integrated with ocr & face_recognition

In [14]:
import cv2
import pytesseract
import face_recognition
import re
import os
import csv
import numpy as np
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import messagebox, filedialog
from tkinter import ttk
import math
import time

#YF 
import qrcode
import smtplib
import socket
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime, timedelta
import random

# Configure tesseract path
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Email configuration
SENDER_EMAIL = "paradisebookstore888@gmail.com"
SENDER_PASSWORD = " iczp fdcl yzyr xtfs"  # Use App Password if Gmail

# Track failed attempts & lock status
failed_attempts = {}  # { email: {"count": int, "locked_until": datetime } }

# Global current image
current_image = None
verification_codes = {}  # Store verification codes temporarily

# ----------------- OCR & ID CLEANUP -----------------
def clean_student_id(ocr_id):
    return (
        ocr_id.upper()
        .replace("O9", "09")
        .replace("O", "0", 1)
        .replace("I", "1")
        .replace("S", "5")
    )

def extract_name_and_id(image_pil):
    img_cv = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    sharpened = cv2.addWeighted(gray, 1.5, blur, -0.5, 0)
    thresh = cv2.adaptiveThreshold(sharpened, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                   cv2.THRESH_BINARY, 15, 10)
    text = pytesseract.image_to_string(thresh)
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    name = ''
    student_id = ''

    for i, line in enumerate(lines):
        line_cleaned = line.replace("O", "0").replace("S", "5")
        match = re.search(r'\d{2}[A-Z0-9]{3}\d{5}', line_cleaned)
        if match:
            raw_id = match.group()
            student_id = raw_id.replace("0", "O", 1)
            for j in range(max(0, i - 3), i):
                candidate = lines[j]
                if (not re.search(r'\d', candidate)
                    and len(candidate.split()) >= 2
                    and candidate.isupper()
                    and not any(k in candidate for k in ['DATE', 'EXPIRY', 'STUDENT', 'TARUMT'])):
                    name = candidate
                    break
            break
    return name.strip(), student_id.strip()

# ----------------- QR CODE GENERATION -----------------
def generate_qr_code(student_name, student_id, email, folder_name):
    qr_data = f"Name: {student_name}\nID: {student_id}\nEmail: {email}"
    qr = qrcode.QRCode(version=1, box_size=10, border=5)
    qr.add_data(qr_data)
    qr.make(fit=True)
    img_qr = qr.make_image(fill_color="black", back_color="white")
    qr_path = os.path.join(folder_name, f"{student_name}.{student_id}_qr.png")
    img_qr.save(qr_path)
    return qr_path

# ----------------- EMAIL VALIDATION -----------------
def is_valid_email(email):
    if not re.match(r"[^@]+@[^@]+\.[^@]+", email):
        return False
    domain = email.split("@")[1]
    try:
        socket.gethostbyname(domain)
        return True
    except socket.error:
        return False

# ----------------- EMAIL SENDING -----------------
def send_email_with_qr(to_email, student_name, qr_path):
    try:
        subject = "🎓 Your Graduation QR Code"
        body = f"Dear {student_name},\n\nPlease find attached your unique QR code for the graduation ceremony.\nBring this QR code with you for scanning during the event.\n\nRegards,\nGraduation Committee"

        msg = MIMEMultipart()
        msg["From"] = SENDER_EMAIL
        msg["To"] = to_email
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))

        with open(qr_path, "rb") as f:
            mime = MIMEBase("image", "png", filename=os.path.basename(qr_path))
            mime.add_header("Content-Disposition", "attachment", filename=os.path.basename(qr_path))
            mime.add_header("X-Attachment-Id", "0")
            mime.add_header("Content-ID", "<0>")
            mime.set_payload(f.read())
            encoders.encode_base64(mime)
            msg.attach(mime)

        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        server.quit()

        messagebox.showinfo("Success", f"QR Code sent to {to_email} successfully!")
        return True

    except Exception as e:
        messagebox.showerror("Email Error", f"Failed to send email: {e}")
        return False

# ----------------- EMAIL VERIFICATION -----------------
def send_verification_code(to_email):
    # 🔒 Check if email is locked BEFORE sending
    if to_email in failed_attempts:
        info = failed_attempts[to_email]
        if info["count"] >= 3 and datetime.now() < info["locked_until"]:
            messagebox.showerror(
                "Locked",
                f"Too many failed attempts for {to_email}. "
                f"Try again after {info['locked_until'].strftime('%H:%M:%S')}."
            )
            return False  # 🚫 don't send email

    try:
        code = str(random.randint(100000, 999999))
        verification_codes[to_email] = code

        subject = "🎓 Your Email Verification Code"
        body = f"Dear Student,\n\nYour verification code for convocation registration is: {code}\n\nDo not share this code with anyone."

        msg = MIMEMultipart()
        msg["From"] = SENDER_EMAIL
        msg["To"] = to_email
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))

        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        server.quit()

        messagebox.showinfo("Verification Sent", f"A verification code has been sent to {to_email}.")
        return True

    except Exception as e:
        messagebox.showerror("Email Error", f"Failed to send verification code: {e}")
        return False

# ----------------- GUI FUNCTIONS -----------------
def display_image_pil(pil_img):
    global current_image
    current_image = pil_img
    img_resized = pil_img.resize((350, 250))
    img_tk = ImageTk.PhotoImage(img_resized)
    panel.config(image=img_tk)
    panel.image = img_tk
    confirm_btn.pack(pady=10)
    retake_btn.pack()
    upload_btn.pack_forget()
    capture_btn.pack_forget()

def upload_image():
    file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
    if file_path:
        pil_img = Image.open(file_path)
        display_image_pil(pil_img)

def take_picture():
    cap = cv2.VideoCapture(0)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    rect_w, rect_h = 480, 300
    rect_x = (frame_width - rect_w) // 2
    rect_y = (frame_height - rect_h) // 2
    captured_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.putText(frame, "Press SPACE to capture, ESC to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
        cv2.imshow("Capture ID Card", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == 32:
            captured_frame = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            break
        elif key == 27:
            break
    cap.release()
    cv2.destroyAllWindows()
    if captured_frame is not None:
        pil_img = Image.fromarray(cv2.cvtColor(captured_frame, cv2.COLOR_BGR2RGB))
        display_image_pil(pil_img)

def confirm_image():
    global current_image
    if current_image is None:
        messagebox.showwarning("No Image", "Please select or capture an image first.")
        return

    name, student_id = extract_name_and_id(current_image)
    if not name and not student_id:
        messagebox.showwarning("OCR Failed", "Could not extract name or student ID.")
        return

    edit_window = tk.Toplevel(root)
    edit_window.title("Confirm Student Info")
    ttk.Label(edit_window, text="Please confirm or edit your information below:").grid(row=0, column=0, columnspan=2, pady=(10, 5))

    def force_uppercase_name(*args):
        current = name_var.get()
        name_var.set(current.upper())

    name_var = tk.StringVar()
    name_var.trace_add("write", force_uppercase_name)

    def force_uppercase_id(*args):
        current = id_var.get()
        id_var.set(current.upper())

    id_var = tk.StringVar()
    id_var.trace_add("write", force_uppercase_id)

    def force_lowercase_email(*args):
        current = email_var.get()
        email_var.set(current.lower())

    email_var = tk.StringVar()
    email_var.trace_add("write", force_lowercase_email)

    ttk.Label(edit_window, text="Name:").grid(row=1, column=0, padx=10, pady=5, sticky="e")
    name_entry = ttk.Entry(edit_window, width=40, textvariable=name_var)
    name_var.set(name)
    name_entry.grid(row=1, column=1, padx=10, pady=5)

    ttk.Label(edit_window, text="Student ID:").grid(row=2, column=0, padx=10, pady=5, sticky="e")
    id_entry = ttk.Entry(edit_window, width=40, textvariable=id_var)
    corrected_id = clean_student_id(student_id)
    id_var.set(corrected_id)
    id_entry.grid(row=2, column=1, padx=10, pady=5)

    ttk.Label(edit_window, text="School Email:").grid(row=3, column=0, padx=10, pady=5, sticky="e")
    email_entry = ttk.Entry(edit_window, width=40, textvariable=email_var)
    email_entry.grid(row=3, column=1, padx=10, pady=5)
    email_entry.focus_set()

    # ----------------- SAVE DATA -----------------
    def save_data():
        final_name = name_entry.get().strip().replace(" ", "_")
        final_id = id_entry.get().strip()
        email = email_entry.get().strip()

        if not final_name or not final_id or not email:
            messagebox.showerror("Error", "Name, Student ID, and Email are required.")
            return

        # Enforce school email
        school_email_pattern = r"^[a-z0-9]+-[a-z]{2}[0-9]{2}@student\.tarc\.edu\.my$"
        if not re.match(school_email_pattern, email.lower()):
            messagebox.showerror("Invalid Email", "Please enter a valid school email (e.g., tanyf-wm22@student.tarc.edu.my).")
            email_entry.focus_set()
            return

        # --- EMAIL VERIFICATION CHECK ---
        file_exists = os.path.exists("student_records.csv")
        if file_exists:
            with open("student_records.csv", "r", newline="") as fr:
                existing_emails = [row["Email"] for row in csv.DictReader(fr)]
            if email in existing_emails:
                messagebox.showerror("Email Already Used", 
                                    "This email has already been verified.\nPlease use a different school email.")
                email_entry.focus_set()  # Go back to email entry
                return  # Stop registration here
       
        # --- SEND VERIFICATION EMAIL ---
        if not send_verification_code(email):
            edit_window.destroy()      # Close confirm window
            retake_or_reselect()
            return  # stop if locked or failed to send
        
        # Ask for verification code
        def ask_verification_code():
            # check if this email is currently locked
            if email in failed_attempts and failed_attempts[email]["count"] >= 3:
                locked_until = failed_attempts[email]["locked_until"]
                if datetime.now() < locked_until:
                    messagebox.showerror(
                        "Locked",
                        f"Too many failed attempts. Try again after {locked_until.strftime('%H:%M:%S')}."
                    )
                    edit_window.destroy()      # ❌ close the confirm ID window
                    retake_or_reselect()       # ✅ back to homepage (Upload / Capture)
                    return  # stop completely

            code_window = tk.Toplevel(edit_window)
            code_window.title("Email Verification")
            ttk.Label(code_window, text=f"A verification code was sent to {email}").pack(pady=10)
            code_var = tk.StringVar()
            ttk.Entry(code_window, textvariable=code_var).pack(pady=5)

            def verify_code():
                user_code = code_var.get().strip()
                correct_code = verification_codes.get(email)

                if user_code == correct_code:
                    # Reset attempts on success
                    failed_attempts[email] = {"count": 0, "locked_until": datetime.min}
                    messagebox.showinfo("Verified", "Email verified successfully!")
                    code_window.destroy()
                    messagebox.showinfo("Next Step", "Now we will capture your face using the webcam.")
                    proceed_with_registration(final_name, final_id, email, edit_window)

                else:
                    # ❌ Wrong code
                    if email not in failed_attempts:
                        failed_attempts[email] = {"count": 0, "locked_until": datetime.min}

                    failed_attempts[email]["count"] += 1
                    remaining = 3 - failed_attempts[email]["count"]

                    if remaining > 0:
                        messagebox.showerror(
                            "Incorrect Code",
                            f"Wrong code. {remaining} attempt(s) left."
                        )
                    else:
                        # lock for 5 minutes
                        failed_attempts[email]["locked_until"] = datetime.now() + timedelta(minutes=5)
                        messagebox.showerror(
                            "Locked",
                            "Too many failed attempts. This email is locked for 5 minutes."
                        )
                        code_window.destroy()
                        edit_window.destroy()  # cancel registration too
                        retake_or_reselect()

            ttk.Button(code_window, text="Verify", command=verify_code).pack(pady=10)
            code_window.grab_set()
        ask_verification_code()
    ttk.Button(edit_window, text="Confirm & Save", command=save_data).grid(row=4, column=0, columnspan=2, pady=10)

def proceed_with_registration(final_name, final_id, email, edit_window):
    folder_name = os.path.join("StudentidFolder", f"{final_name}.{final_id}")
    os.makedirs(folder_name, exist_ok=True)
    img_path = os.path.join(folder_name, f"{final_name}.{final_id}.jpg")
    current_image.save(img_path)

    # CSV update
    file_exists = os.path.exists("student_records.csv")
    existing_ids = []
    if file_exists:
        with open("student_records.csv", "r", newline="") as fr:
            existing_ids = [row["Student ID"] for row in csv.DictReader(fr)]
    if final_id not in existing_ids:
        with open("student_records.csv", "a", newline="") as f:
            fieldnames = ["Name", "Student ID", "Email", "Image Path"]
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists or os.stat("student_records.csv").st_size == 0:
                writer.writeheader()
            writer.writerow({
                "Name": final_name.replace("_", " "),
                "Student ID": final_id,
                "Email": email,
                "Image Path": img_path
            })
    else:
        messagebox.showinfo("Info", f"Student {final_name} ({final_id}) already exists. Updated image path and email.")

    # --- FACE CAPTURE ---
    temp_folder_created = False
    frame_count = 0
    buffer_encodings = []
    capture_interval = 10
    timeout_seconds = 60
    start_time = time.time()
    cap = cv2.VideoCapture(0)
    from mtcnn import MTCNN
    detector = MTCNN()

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = detector.detect_faces(rgb)
        h, w, _ = frame.shape
        center = (w // 2, h // 2)
        radius = 140
        segments = 40
        completed_ratio = len(buffer_encodings) / 7

        # Draw progress ring segments
        for i in range(segments):
            angle = 2 * math.pi * i / segments
            x1 = int(center[0] + radius * math.cos(angle))
            y1 = int(center[1] + radius * math.sin(angle))
            x2 = int(center[0] + (radius + 12) * math.cos(angle))
            y2 = int(center[1] + (radius + 12) * math.sin(angle))
            if i < math.floor(completed_ratio * segments):
                cv2.line(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
            else:
                cv2.line(frame, (x1, y1), (x2, y2), (180, 180, 180), 2)

        cv2.circle(frame, center, radius - 10, (255, 255, 255), 2)
        cv2.putText(frame, "Move your head slowly to complete the circle", (center[0] - 200, center[1] + radius + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        # Face detection & encoding collection
        if len(results) == 1:
            if not temp_folder_created:
                os.makedirs(folder_name, exist_ok=True)
                temp_folder_created = True
                img_path = os.path.join(folder_name, f"{final_name}.{final_id}.jpg")
                current_image.save(img_path)
            if frame_count % capture_interval == 0:
                x, y, w_box, h_box = results[0]['box']
                top, right, bottom, left = y, x + w_box, y + h_box, x
                encodings = face_recognition.face_encodings(rgb, known_face_locations=[(top, right, bottom, left)])
                if encodings:
                    buffer_encodings.append(encodings[0])
                    if len(buffer_encodings) >= 7:
                        mean_encoding = np.mean(buffer_encodings, axis=0)
                        np.save(os.path.join(folder_name, "face_encoding.npy"), mean_encoding)
                        with open("student_records.csv", "a", newline="") as f:
                            writer = csv.writer(f)
                            if os.stat("student_records.csv").st_size == 0:
                                writer.writerow(["Name", "Student ID", "Email", "Image Path"])
                            writer.writerow([final_name.replace("_", " "), final_id, email, img_path])
                        messagebox.showinfo("Success", "Registration complete! You may now proceed to register the next student.")
                        break
        elif len(results) > 1:
            cv2.putText(frame, "Multiple faces detected", (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        frame_count += 1
        if time.time() - start_time > timeout_seconds:
            if temp_folder_created:
                try:
                    for f in os.listdir(folder_name):
                        os.remove(os.path.join(folder_name, f))
                    os.rmdir(folder_name)
                except:
                    pass
            messagebox.showwarning("Timeout", "No face detected in time. Registration failed. Please try again.")
            break

        cv2.imshow("Live Face Capture", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    edit_window.destroy()
    retake_or_reselect()


    # --- QR GENERATION & EMAIL AFTER FACE SUCCESS ---
    qr_path = generate_qr_code(final_name, final_id, email, folder_name)
    email_sent = send_email_with_qr(email, final_name, qr_path)
    if email_sent:
        messagebox.showinfo("Success", "Registration complete!")
        edit_window.destroy()
        retake_or_reselect()

def retake_or_reselect():
    global current_image
    panel.config(image=None)
    panel.image = None
    current_image = None
    confirm_btn.pack_forget()
    retake_btn.pack_forget()
    upload_btn.pack(pady=10)
    capture_btn.pack(pady=5)

# ----------------- GUI SETUP -----------------
root = tk.Tk()
root.title("🎓 Convocation Registration System")
root.geometry("480x600")

welcome = ttk.Label(root, text="Welcome to Convocation Registration System!", font=("Helvetica", 14, "bold"))
welcome.pack(pady=10)

instruction = ttk.Label(root, text="Please register using your student ID card (upload or webcam)")
instruction.pack()

upload_btn = ttk.Button(root, text="Upload Student ID Card", command=upload_image)
upload_btn.pack(pady=10)

capture_btn = ttk.Button(root, text="Capture Student ID via Webcam", command=take_picture)
capture_btn.pack(pady=5)

panel = ttk.Label(root)
panel.pack(padx=10, pady=10)

confirm_btn = ttk.Button(root, text="Confirm ID Card", command=confirm_image)
retake_btn = ttk.Button(root, text="Retake / Select Another Image", command=retake_or_reselect)

root.mainloop()


DURING CEREMONY QR + FACE

In [15]:
import cv2
import numpy as np
import face_recognition
import csv
from pyzbar import pyzbar
from tkinter import Tk, messagebox
import time
import os
import pyttsx3

# ----------------- Helper for popups -----------------
def show_popup(title, message):
    root = Tk()
    root.withdraw()
    messagebox.showinfo(title, message)
    root.destroy()

# ----------------- Load students from student records -----------------
student_encodings = {}
student_records_file = "student_records.csv"
students_list = []

with open(student_records_file, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row['Name'].replace(" ", "_")
        sid = row['Student ID']
        image_path = row['Image Path']
        folder_key = f"{name}.{sid}"
        if os.path.exists(image_path):
            img = face_recognition.load_image_file(image_path)
            encoding = face_recognition.face_encodings(img)[0]
            student_encodings[folder_key] = encoding
            students_list.append({'name': name, 'sid': sid, 'key': folder_key})
        else:
            print(f"Image not found for {folder_key}, skipping.")

if not student_encodings:
    print("=== No registered student encodings found. ===")
    exit()

# ----------------- Setup attendance file -----------------
attendance_file = "attendance.csv"

# Always preload all students as Absent
if not os.path.exists(attendance_file):
    # Create attendance file with all students Absent
    with open(attendance_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Name", "Student ID", "Status"])
        for student in students_list:
            writer.writerow([student['name'].replace("_", " "), student['sid'], "Absent"])
else:
    print("📄 Using existing attendance.csv (will keep previous marks).")
# Keep track of already marked students
attendance_marked_students = set()
with open(attendance_file, "r") as f:
    next(f)  # skip header
    for line in f:
        cols = line.strip().split(",")
        if len(cols) >= 3 and cols[2] == "Present":
            attendance_marked_students.add(f"{cols[0].replace(' ','_')}.{cols[1]}")


# ----------------- TTS --------------------------
def talk_student_name(student_name):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 0.8)
        print(f"Speaking: {student_name}")
        engine.say(student_name)
        engine.runAndWait()
    finally:
        if 'engine' in locals():
            engine.stop()
# ----------------- Start webcam -----------------
cap = cv2.VideoCapture(0)
font = cv2.FONT_HERSHEY_SIMPLEX

qr_verified_student = None
current_step = "QR Scan"

face_match_start_time = None
countdown_time = 3  # seconds needed to confirm face

last_spoken = ""
speak_time = 0  

attendenceMarkedDone = False
print("📷 Webcam started. Please scan your QR code inside the box.")

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w, _ = frame.shape
    y_offset = 30

    # --- Define boxes ---
    # QR Scan box (smaller)
    qr_box_h = h // 3
    qr_box_w = w // 3
    qr_box_top = (h - qr_box_h) // 2
    qr_box_bottom = qr_box_top + qr_box_h
    qr_box_left = (w - qr_box_w) // 2
    qr_box_right = qr_box_left + qr_box_w

    # Face Recognition box (bigger)
    face_box_h = h * 2 // 3
    face_box_w = w * 2 // 3
    face_box_top = (h - face_box_h) // 2
    face_box_bottom = face_box_top + face_box_h
    face_box_left = (w - face_box_w) // 2
    face_box_right = face_box_left + face_box_w

    # Draw boxes
    if current_step == "QR Scan":
        cv2.rectangle(frame, (qr_box_left, qr_box_top), (qr_box_right, qr_box_bottom), (0, 255, 255), 2)
    # else:
        # cv2.rectangle(frame, (face_box_left, face_box_top), (face_box_right, face_box_bottom), (255, 255, 0), 2)

    # ===== QR SCANNING PHASE =====
    if current_step == "QR Scan" and qr_verified_student is None:
        cv2.putText(frame, "Align QR inside the box", (qr_box_left, qr_box_top - 10),
                    font, 0.6, (0, 255, 255), 2)

        decoded_qrs = pyzbar.decode(frame)
        for qr in decoded_qrs:
            (x, y, qr_w, qr_h) = qr.rect
            qr_center_x = x + qr_w // 2
            qr_center_y = y + qr_h // 2

            if qr_box_left < qr_center_x < qr_box_right and qr_box_top < qr_center_y < qr_box_bottom:
                qr_data = qr.data.decode("utf-8")
                lines = qr_data.split("\n")
                try:
                    name_line = [l for l in lines if "Name:" in l][0]
                    id_line = [l for l in lines if "ID:" in l][0]
                    student_name = name_line.split(":")[1].strip().replace(" ", "_")
                    student_id = id_line.split(":")[1].strip()
                    folder_key = f"{student_name}.{student_id}"

                    if folder_key in student_encodings:
                        if folder_key in attendance_marked_students:
                            show_popup("Already Taken",
                                       f"Attendance already marked for {student_name.replace('_',' ')} ({student_id})")
                        else:
                            qr_verified_student = folder_key
                            current_step = "Face Scan"
                            show_popup("QR Verified", f"QR Verified!\n{student_name} ({student_id})")
                    else:
                        show_popup("Invalid QR Code", "This student is not in the database!")
                except:
                    show_popup("Invalid QR Code", "QR Code format is not valid!")

    # ===== FACE RECOGNITION PHASE =====
    elif current_step == "Face Scan" and qr_verified_student:
        face_locations = face_recognition.face_locations(rgb)
        face_encodings = face_recognition.face_encodings(rgb, face_locations)

        matched = False
        for (top, right, bottom, left), live_encoding in zip(face_locations, face_encodings):
            face_center_x = (left + right) // 2
            face_center_y = (top + bottom) // 2
            if not (face_box_left < face_center_x < face_box_right and face_box_top < face_center_y < face_box_bottom):
                cv2.putText(frame, "Keep face inside the box!", (10, h - 20), font, 0.7, (0, 0, 255), 2)
                continue

            db_encoding = student_encodings[qr_verified_student]
            distance = face_recognition.face_distance([db_encoding], live_encoding)[0]

            if distance < 0.6:
                cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 3)
                cv2.putText(frame, "Face Match", (left, top - 10), font, 0.7, (0, 255, 0), 2)
                matched = True
                
            else:
                cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 3)
                cv2.putText(frame, "Not matching!", (left, top - 10), font, 0.6, (0, 0, 255), 2)


            

        # Handle countdown if matched
        if matched:
            if face_match_start_time is None:
                face_match_start_time = time.time()
            elapsed = int(time.time() - face_match_start_time)
            remaining = countdown_time - elapsed

            if remaining > 0:
                cv2.putText(frame, f"Hold still... {remaining}", (50, h - 50), font, 0.8, (0, 255, 0), 2)
            else:
                # Update attendance to Present
                
                name, sid = qr_verified_student.split(".", 1)
                rows = []
                with open(attendance_file, "r") as f:
                    reader = csv.reader(f)
                    rows = list(reader)
                with open(attendance_file, "w", newline="") as f:
                    writer = csv.writer(f)
                    for row in rows:
                        if len(row) >= 3 and row[0].replace(" ", "_") == name and row[1] == sid:
                            row[2] = "Present"
                        writer.writerow(row)

                attendance_marked_students.add(qr_verified_student)
                show_popup("Attendance Marked", f"Attendance marked for {name.replace('_',' ')} ({sid})")

                qr_verified_student = None
                current_step = "QR Scan"
                face_match_start_time = None
                attendenceMarkedDone = True
        else:
            face_match_start_time = None

    # ===== DISPLAY STATUS =====
    status_text = "Step: " + current_step
    cv2.putText(frame, status_text, (10, y_offset), font, 0.7, (0, 0, 0), 2)
    
    if attendenceMarkedDone == True:
        student_name_replace = student_name.replace('_', ' ')
        if student_name_replace != last_spoken:
                last_spoken = student_name_replace
                speak_time = time.time() + 15 

    if qr_verified_student:
        name, sid = qr_verified_student.split(".", 1)
        cv2.putText(frame, f"{name.replace('_',' ')} - {sid}", (10, y_offset + 30), font, 0.7, (0, 255, 0), 2)

    if last_spoken and time.time() >= speak_time and speak_time > 0:
        talk_student_name(last_spoken)
        speak_time = 0
        attendenceMarkedDone = False
        
    cv2.putText(frame, "Press ESC to quit", (10, y_offset + 70), font, 0.6, (0, 0, 0), 2)
    cv2.imshow("Student Check-in System", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

📄 Using existing attendance.csv (will keep previous marks).
📷 Webcam started. Please scan your QR code inside the box.
Speaking: WONG WAI FENG
